# LeetCode 760 — Find Anagram Mappings bằng Python

**Link đề:** https://leetcode.com/problems/find-anagram-mappings/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def anagramMappings(self, nums1: list[int], nums2: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu "**anagram**" của mảng (cùng các phần tử, cùng số lần, khác thứ tự) và "**mảng ánh xạ chỉ số**".
2. Ôn `list.index`, `dict`, `defaultdict(list)`, `enumerate`, `sorted(..., key=...)`.
3. Đi từ cách trực giác `O(n²)` (`index` cho từng phần tử) → **bảng băm giá trị → chỉ số** `O(n)`.
4. Xử lý **phần tử trùng**: ánh xạ "hợp lệ" vs ánh xạ là **hoán vị** (mỗi chỉ số dùng đúng một lần).
5. Viết **hàm kiểm tra đáp án** (validator) vì đề chấp nhận nhiều đáp án — kỹ năng test quan trọng.

## 1. Đề bài

Cho hai mảng số nguyên `nums1` và `nums2`, trong đó `nums2` là **anagram** của `nums1`: `nums2` được tạo bằng cách **xáo trộn thứ tự** các phần tử của `nums1`. Hai mảng **có thể chứa phần tử trùng**.

Trả về mảng ánh xạ `mapping` cùng độ dài, trong đó `mapping[i] = j` nghĩa là phần tử thứ `i` của `nums1` xuất hiện trong `nums2` tại chỉ số `j`, tức là:

```text
nums2[mapping[i]] == nums1[i]   với mọi i
```

Nếu có nhiều đáp án, trả về **đáp án bất kỳ**.

**Ví dụ 1:** `nums1 = [12,28,46,32,50]`, `nums2 = [50,12,32,46,28]` → `[1,4,3,2,0]`.
`mapping[0] = 1` vì `nums1[0] = 12` nằm ở `nums2[1]`; `mapping[1] = 4` vì `28` nằm ở `nums2[4]`; ...

**Ví dụ 2:** `nums1 = [84,46]`, `nums2 = [84,46]` → `[0,1]`.

**Constraints (theo hiểu biết):** `1 <= len(nums1) <= 100`, `len(nums2) == len(nums1)`, `0 <= nums1[i], nums2[i] <= 10^5`, `nums2` là anagram của `nums1`.

In [1]:
nums1 = [12, 28, 46, 32, 50]
nums2 = [50, 12, 32, 46, 28]
mapping = [1, 4, 3, 2, 0]
for i, j in enumerate(mapping):
    print(f"i={i}: nums1[{i}]={nums1[i]:>2} -> mapping={j} -> nums2[{j}]={nums2[j]:>2}  khớp: {nums1[i] == nums2[j]}")

i=0: nums1[0]=12 -> mapping=1 -> nums2[1]=12  khớp: True
i=1: nums1[1]=28 -> mapping=4 -> nums2[4]=28  khớp: True
i=2: nums1[2]=46 -> mapping=3 -> nums2[3]=46  khớp: True
i=3: nums1[3]=32 -> mapping=2 -> nums2[2]=32  khớp: True
i=4: nums1[4]=50 -> mapping=0 -> nums2[0]=50  khớp: True


### Hai cách hiểu khi có phần tử trùng

`nums1 = [1, 1]`, `nums2 = [1, 1]`:
- `[0, 0]` thỏa công thức `nums2[mapping[i]] == nums1[i]` — nhưng chỉ số 0 bị dùng **hai lần**.
- `[0, 1]` hoặc `[1, 0]` vừa thỏa công thức vừa là **hoán vị** (mỗi chỉ số của `nums2` dùng đúng một lần) — "đẹp" hơn về ý nghĩa ghép cặp.

Theo hiểu biết, LeetCode chấp nhận mọi `mapping` thỏa công thức (kể cả dùng lại chỉ số). Notebook sẽ dạy **cả hai**: bản đơn giản và bản hoán vị (an toàn với mọi cách chấm).

## 2. Nền tảng Python

### 2.1. `list.index(x)` — tìm chỉ số đầu tiên

`nums2.index(x)` quét từ đầu, trả chỉ số **đầu tiên** có giá trị `x` — tốn `O(n)`. Ném `ValueError` nếu không có.

In [2]:
nums2 = [50, 12, 32, 46, 28]
print(nums2.index(46))
print([1, 7, 1].index(1))      # luôn trả 0 với phần tử trùng

3
0


### 2.2. `dict` — tra cứu giá trị → chỉ số trong `O(1)` trung bình

Duyệt `nums2` một lần, lưu `pos[value] = index`. Sau đó `pos[x]` trả ngay chỉ số của `x`.

In [3]:
pos = {}
for j, v in enumerate([50, 12, 32, 46, 28]):
    pos[v] = j
print(pos)
print(pos[46], 46 in pos, 99 in pos)

{50: 0, 12: 1, 32: 2, 46: 3, 28: 4}
3 True False


Với phần tử trùng, phép gán sau **ghi đè** phép gán trước → `pos` giữ chỉ số **cuối cùng**. Cách viết gọn bằng **dict comprehension**: `{v: j for j, v in enumerate(nums2)}`.

In [4]:
print({v: j for j, v in enumerate([1, 7, 1])})   # {1: 2, 7: 1} -> chỉ số cuối của 1

{1: 2, 7: 1}


### 2.3. `defaultdict(list)` — giá trị → **danh sách** chỉ số

Để không mất chỉ số khi trùng, lưu **tất cả**: `pos[v].append(j)`. `defaultdict(list)` tự tạo list rỗng cho key mới.

In [5]:
from collections import defaultdict

pos = defaultdict(list)
for j, v in enumerate([1, 7, 1, 1]):
    pos[v].append(j)
print(dict(pos))
print(pos[1].pop(), pos[1].pop(), pos[1])   # pop() lấy và XÓA phần tử cuối

{1: [0, 2, 3], 7: [1]}
3 2 [0]


### 2.4. `sorted(range(n), key=...)` — sắp xếp **chỉ số** theo giá trị (argsort)

`sorted(range(n), key=lambda i: a[i])` trả các chỉ số theo thứ tự giá trị tăng dần. Đây là "argsort" — dùng cho Cách 4.

In [6]:
a = [50, 12, 32, 46, 28]
order = sorted(range(len(a)), key=lambda i: a[i])
print(order)                    # chỉ số của phần tử nhỏ nhất, nhì, ...
print([a[i] for i in order])    # giá trị tăng dần

[1, 4, 2, 3, 0]
[12, 28, 32, 46, 50]


## 3. Viết hàm kiểm tra đáp án trước (validator)

Vì đề cho phép **nhiều đáp án**, ta **không** test bằng `assert result == [1, 4, 3, 2, 0]` cứng nhắc. Thay vào đó kiểm **tính chất**:
1. Cùng độ dài với `nums1`.
2. Mọi `mapping[i]` là chỉ số hợp lệ của `nums2`.
3. `nums2[mapping[i]] == nums1[i]` với mọi `i`.
4. (Tùy chọn, chế độ nghiêm) `mapping` là **hoán vị** của `0..n-1`.

In [7]:
def is_valid_mapping(nums1, nums2, mapping, require_permutation=False):
    n = len(nums1)
    if len(mapping) != n:
        return False
    for i, j in enumerate(mapping):
        if not (0 <= j < len(nums2)) or nums2[j] != nums1[i]:
            return False
    if require_permutation and sorted(mapping) != list(range(n)):
        return False
    return True

print(is_valid_mapping([12, 28, 46, 32, 50], [50, 12, 32, 46, 28], [1, 4, 3, 2, 0], True))
print(is_valid_mapping([1, 1], [1, 1], [0, 0]), is_valid_mapping([1, 1], [1, 1], [0, 0], True))
print(is_valid_mapping([1, 2], [2, 1], [0, 1]))

True
True False
False


`sorted(mapping) != list(range(n))` — một mảng là hoán vị của `0..n-1` ⟺ sắp xếp xong thì đúng bằng `[0, 1, ..., n-1]`.

## 4. Cách 1 — Trực giác: `nums2.index(x)` cho từng phần tử

In [8]:
def mapping_index(nums1, nums2):
    return [nums2.index(x) for x in nums1]

r = mapping_index([12, 28, 46, 32, 50], [50, 12, 32, 46, 28])
print(r, is_valid_mapping([12, 28, 46, 32, 50], [50, 12, 32, 46, 28], r))
r = mapping_index([1, 1], [1, 1])
print(r, "hợp lệ:", is_valid_mapping([1, 1], [1, 1], r), "| hoán vị:", is_valid_mapping([1, 1], [1, 1], r, True))

[1, 4, 3, 2, 0] True
[0, 0] hợp lệ: True | hoán vị: False


### Nút thắt

- Mỗi `index` quét tới `n` phần tử → tổng **`O(n²)`**. Với `n = 100` vẫn nhanh, nhưng ta đang **quét lại** `nums2` từ đầu cho mỗi phần tử.
- Với phần tử trùng, mọi bản sao đều về **cùng một** chỉ số (không phải hoán vị).

**Insight:** câu hỏi "giá trị `x` nằm ở đâu trong `nums2`?" được hỏi `n` lần trên **cùng** một mảng → **tiền xử lý một lần** thành bảng băm `giá trị → chỉ số`, mỗi câu hỏi còn `O(1)` trung bình.

## 5. Cách 2 — Bảng băm giá trị → chỉ số

In [9]:
def mapping_dict(nums1, nums2):
    pos = {v: j for j, v in enumerate(nums2)}
    return [pos[x] for x in nums1]

r = mapping_dict([12, 28, 46, 32, 50], [50, 12, 32, 46, 28])
print(r, is_valid_mapping([12, 28, 46, 32, 50], [50, 12, 32, 46, 28], r, True))
r = mapping_dict([1, 1], [1, 1])
print(r, "hợp lệ:", is_valid_mapping([1, 1], [1, 1], r), "| hoán vị:", is_valid_mapping([1, 1], [1, 1], r, True))

[1, 4, 3, 2, 0] True
[1, 1] hợp lệ: True | hoán vị: False


`O(n)` — đủ để qua LeetCode (theo hiểu biết, đáp án dùng lại chỉ số vẫn được chấp nhận). Nhưng với phần tử trùng, nó vẫn không phải hoán vị. Nếu đề/người chấm yêu cầu **ghép cặp một-một**, ta cần Cách 3.

## 6. Cách 3 — Giá trị → danh sách chỉ số, `pop` khi dùng (lời giải chính)

Ghép từng đoạn:

**Đoạn A — tiền xử lý:** `pos[v]` = list mọi chỉ số `j` có `nums2[j] == v`.

**Đoạn B — ghép:** với mỗi `x` trong `nums1`, lấy **và xóa** một chỉ số khỏi `pos[x]` bằng `pop()` → mỗi chỉ số của `nums2` được dùng **đúng một lần**.

Vì `nums2` là anagram của `nums1`, số lần `x` xuất hiện ở hai mảng bằng nhau → `pos[x]` không bao giờ rỗng khi cần `pop`.

In [10]:
def mapping_bijective(nums1, nums2):
    pos = defaultdict(list)                 # Đoạn A
    for j, v in enumerate(nums2):
        pos[v].append(j)
    return [pos[x].pop() for x in nums1]    # Đoạn B

for a, c in [([12, 28, 46, 32, 50], [50, 12, 32, 46, 28]), ([84, 46], [84, 46]), ([1, 1], [1, 1]), ([5, 3, 5, 5], [5, 5, 3, 5])]:
    r = mapping_bijective(a, c)
    print(a, c, "->", r, "| hoán vị hợp lệ:", is_valid_mapping(a, c, r, True))

[12, 28, 46, 32, 50] [50, 12, 32, 46, 28] -> [1, 4, 3, 2, 0] | hoán vị hợp lệ: True
[84, 46] [84, 46] -> [0, 1] | hoán vị hợp lệ: True
[1, 1] [1, 1] -> [1, 0] | hoán vị hợp lệ: True
[5, 3, 5, 5] [5, 5, 3, 5] -> [3, 2, 1, 0] | hoán vị hợp lệ: True


`pop()` lấy phần tử **cuối** list trong `O(1)`. (`pop(0)` lấy phần tử đầu nhưng tốn `O(k)` — tránh dùng trong vòng lặp.)

## 7. Trace từng bước

`nums1 = [5, 3, 5, 5]`, `nums2 = [5, 5, 3, 5]`.

Đoạn A: `pos = {5: [0, 1, 3], 3: [2]}`.

| i | x = nums1[i] | pos[x] trước | pop() → mapping[i] | pos[x] sau |
|---:|---:|---|---:|---|
| 0 | 5 | [0, 1, 3] | 3 | [0, 1] |
| 1 | 3 | [2] | 2 | [] |
| 2 | 5 | [0, 1] | 1 | [0] |
| 3 | 5 | [0] | 0 | [] |

Kết quả `[3, 2, 1, 0]`: kiểm `nums2[3]=5, nums2[2]=3, nums2[1]=5, nums2[0]=5` ✔, và là hoán vị ✔.

In [11]:
def trace_mapping(nums1, nums2):
    pos = defaultdict(list)
    for j, v in enumerate(nums2):
        pos[v].append(j)
    print("Đoạn A: pos =", dict(pos))
    mapping = []
    for i, x in enumerate(nums1):
        before = list(pos[x])
        j = pos[x].pop()
        mapping.append(j)
        print(f"i={i}: x={x} | pos[x] trước={before} -> lấy {j} | pos[x] sau={pos[x]} | mapping={mapping}")
    return mapping

trace_mapping([5, 3, 5, 5], [5, 5, 3, 5])

Đoạn A: pos = {5: [0, 1, 3], 3: [2]}
i=0: x=5 | pos[x] trước=[0, 1, 3] -> lấy 3 | pos[x] sau=[0, 1] | mapping=[3]
i=1: x=3 | pos[x] trước=[2] -> lấy 2 | pos[x] sau=[] | mapping=[3, 2]
i=2: x=5 | pos[x] trước=[0, 1] -> lấy 1 | pos[x] sau=[0] | mapping=[3, 2, 1]
i=3: x=5 | pos[x] trước=[0] -> lấy 0 | pos[x] sau=[] | mapping=[3, 2, 1, 0]


[3, 2, 1, 0]

In [12]:
trace_mapping([12, 28, 46, 32, 50], [50, 12, 32, 46, 28])

Đoạn A: pos = {50: [0], 12: [1], 32: [2], 46: [3], 28: [4]}
i=0: x=12 | pos[x] trước=[1] -> lấy 1 | pos[x] sau=[] | mapping=[1]
i=1: x=28 | pos[x] trước=[4] -> lấy 4 | pos[x] sau=[] | mapping=[1, 4]
i=2: x=46 | pos[x] trước=[3] -> lấy 3 | pos[x] sau=[] | mapping=[1, 4, 3]
i=3: x=32 | pos[x] trước=[2] -> lấy 2 | pos[x] sau=[] | mapping=[1, 4, 3, 2]
i=4: x=50 | pos[x] trước=[0] -> lấy 0 | pos[x] sau=[] | mapping=[1, 4, 3, 2, 0]


[1, 4, 3, 2, 0]

## 8. Cách 4 — Sắp xếp chỉ số (argsort) và ghép theo thứ hạng

Ý tưởng khác hẳn, không dùng bảng băm: sắp các **chỉ số** của mỗi mảng theo giá trị. Phần tử hạng `k` của `nums1` và phần tử hạng `k` của `nums2` có **cùng giá trị** (vì hai mảng cùng tập phần tử) → ghép chúng.

```text
nums1 = [12, 28, 46, 32, 50]  -> order1 (chỉ số theo giá trị tăng) = [0, 1, 3, 2, 4]
nums2 = [50, 12, 32, 46, 28]  -> order2                              = [1, 4, 2, 3, 0]
ghép: mapping[order1[k]] = order2[k]
```

In [13]:
def mapping_argsort(nums1, nums2):
    n = len(nums1)
    order1 = sorted(range(n), key=lambda i: nums1[i])
    order2 = sorted(range(n), key=lambda j: nums2[j])
    mapping = [0] * n
    for i, j in zip(order1, order2):
        mapping[i] = j
    return mapping

a, c = [12, 28, 46, 32, 50], [50, 12, 32, 46, 28]
print(sorted(range(5), key=lambda i: a[i]), sorted(range(5), key=lambda j: c[j]))
r = mapping_argsort(a, c)
print(r, is_valid_mapping(a, c, r, True))
r = mapping_argsort([5, 3, 5, 5], [5, 5, 3, 5])
print(r, is_valid_mapping([5, 3, 5, 5], [5, 5, 3, 5], r, True))

[0, 1, 3, 2, 4] [1, 4, 2, 3, 0]
[1, 4, 3, 2, 0] True
[0, 2, 1, 3] True


Cách 4 tự nhiên cho ra **hoán vị** (vì `order2` là hoán vị của `0..n-1`, mỗi `j` dùng đúng một lần), và không cần bảng băm. Đổi lại tốn `O(n log n)` cho sắp xếp. Đây cũng là ý tưởng "**ghép theo thứ hạng**" hữu ích ở nhiều bài khác.

## 9. Code nộp LeetCode

### Bản hoán vị với `defaultdict(list)` (khuyến nghị)

In [14]:
from collections import defaultdict


class Solution:
    def anagramMappings(self, nums1: list[int], nums2: list[int]) -> list[int]:
        pos = defaultdict(list)
        for j, v in enumerate(nums2):
            pos[v].append(j)
        return [pos[x].pop() for x in nums1]


sol = Solution()
print(sol.anagramMappings([12, 28, 46, 32, 50], [50, 12, 32, 46, 28]))
print(sol.anagramMappings([84, 46], [84, 46]))

[1, 4, 3, 2, 0]
[0, 1]


Kết quả ví dụ 1 là `[1, 4, 3, 2, 0]` — trùng đáp án mẫu (vì không có phần tử trùng, đáp án là duy nhất).

### Bản `dict` một dòng và bản argsort

Trên LeetCode đổi tên class thành `Solution`. (Một số phiên bản cũ của đề đặt tên tham số là `A, B` — chỉ cần giữ đúng thứ tự.)

In [15]:
class SolutionDict:
    def anagramMappings(self, nums1: list[int], nums2: list[int]) -> list[int]:
        pos = {v: j for j, v in enumerate(nums2)}
        return [pos[x] for x in nums1]


class SolutionArgsort:
    def anagramMappings(self, nums1: list[int], nums2: list[int]) -> list[int]:
        return mapping_argsort(nums1, nums2)


print(SolutionDict().anagramMappings([84, 46], [84, 46]), SolutionArgsort().anagramMappings([84, 46], [84, 46]))

[0, 1] [0, 1]


## 10. Vì sao đúng?

**Cách 2.** Sau tiền xử lý, `pos[v]` là một chỉ số `j` với `nums2[j] == v` (chỉ số cuối cùng). Vì `nums2` là anagram của `nums1`, mọi `x` trong `nums1` đều có trong `nums2` → `pos[x]` tồn tại và `nums2[pos[x]] == x` ✔.

**Cách 3.** Gọi `c(v)` = số lần `v` xuất hiện (bằng nhau ở hai mảng). Invariant: *trước khi xử lý `nums1[i]`, `pos[v]` chứa đúng những chỉ số `j` có `nums2[j] == v` **chưa được dùng**, và số phần tử còn lại = `c(v)` − (số lần `v` đã xuất hiện trong `nums1[0..i-1]`).*
- Mỗi lần gặp `x`, `pos[x]` còn ít nhất 1 phần tử (vì số lần `x` trong `nums1[0..i]` ≤ `c(x)`) → `pop` hợp lệ, trả `j` với `nums2[j] == x` ✔.
- `j` bị xóa khỏi `pos[x]` → không bao giờ dùng lại → mapping là **đơn ánh**; `n` phần tử đơn ánh vào `n` chỉ số → **hoán vị** ✔.

**Cách 4.** Sắp xếp hai mảng cùng tập phần tử cho cùng dãy giá trị tăng dần → hạng `k` của hai bên có cùng giá trị → `nums2[order2[k]] == nums1[order1[k]]` ✔; `order2` là hoán vị → mapping là hoán vị ✔.

## 11. Độ phức tạp

Gọi `n = len(nums1)`.

| Cách | Time | Space phụ | Hoán vị khi trùng? |
|---|---|---|:---:|
| 1. `index` lặp | `O(n²)` | `O(1)` | ❌ |
| 2. `dict` giá trị → chỉ số | `O(n)` trung bình | `O(n)` | ❌ |
| 3. `dict` giá trị → list, `pop` | `O(n)` trung bình | `O(n)` | ✅ |
| 4. Argsort + ghép hạng | `O(n log n)` | `O(n)` | ✅ |

(Không tính `O(n)` của mảng kết quả.) Với `n ≤ 100`, mọi cách đều tức thì; giá trị của bài nằm ở **kỹ thuật tra ngược giá trị → vị trí** và cách xử lý trùng lặp.

## 12. Bộ test

Dùng `is_valid_mapping` thay vì so cứng. Với ví dụ không có phần tử trùng, đáp án là **duy nhất** nên có thể so cứng thêm.

In [16]:
def check_valid(name, nums1, nums2, mapping, strict=False):
    assert is_valid_mapping(nums1, nums2, mapping, strict), (name, nums1, nums2, mapping)
    print(f"PASS  {name}: {mapping}")

sol = Solution()
r = sol.anagramMappings([12, 28, 46, 32, 50], [50, 12, 32, 46, 28])
assert r == [1, 4, 3, 2, 0]
check_valid("đề 1", [12, 28, 46, 32, 50], [50, 12, 32, 46, 28], r, True)
check_valid("đề 2", [84, 46], [84, 46], sol.anagramMappings([84, 46], [84, 46]), True)
check_valid("1 phần tử", [7], [7], sol.anagramMappings([7], [7]), True)
check_valid("toàn trùng", [1, 1, 1], [1, 1, 1], sol.anagramMappings([1, 1, 1], [1, 1, 1]), True)
check_valid("trùng lẫn", [5, 3, 5, 5], [5, 5, 3, 5], sol.anagramMappings([5, 3, 5, 5], [5, 5, 3, 5]), True)
check_valid("đảo ngược", [1, 2, 3, 4], [4, 3, 2, 1], sol.anagramMappings([1, 2, 3, 4], [4, 3, 2, 1]), True)
check_valid("có số 0 và 10^5", [0, 100000, 0], [100000, 0, 0], sol.anagramMappings([0, 100000, 0], [100000, 0, 0]), True)

PASS  đề 1: [1, 4, 3, 2, 0]
PASS  đề 2: [0, 1]
PASS  1 phần tử: [0]
PASS  toàn trùng: [2, 1, 0]
PASS  trùng lẫn: [3, 2, 1, 0]
PASS  đảo ngược: [3, 2, 1, 0]
PASS  có số 0 và 10^5: [2, 0, 1]


In [17]:
import random

rng = random.Random(760)
loose = [mapping_index, mapping_dict, SolutionDict().anagramMappings]
strict = [mapping_bijective, mapping_argsort, Solution().anagramMappings, SolutionArgsort().anagramMappings]
not_perm = 0
for _ in range(3000):
    n = rng.randint(1, 10)
    nums1 = [rng.randint(0, 4) for _ in range(n)]
    nums2 = nums1[:]
    rng.shuffle(nums2)
    for f in loose:
        r = f(nums1, nums2)
        assert is_valid_mapping(nums1, nums2, r)
        not_perm += not is_valid_mapping(nums1, nums2, r, True)
    for f in strict:
        assert is_valid_mapping(nums1, nums2, f(nums1, nums2), True)
print("3000 bộ ngẫu nhiên: mọi cách đều hợp lệ; nhóm 'strict' luôn cho hoán vị.")
print("Số lần nhóm 'loose' cho đáp án hợp lệ nhưng KHÔNG phải hoán vị:", not_perm)

3000 bộ ngẫu nhiên: mọi cách đều hợp lệ; nhóm 'strict' luôn cho hoán vị.
Số lần nhóm 'loose' cho đáp án hợp lệ nhưng KHÔNG phải hoán vị: 6804


`nums1[:]` tạo **bản sao** (để `shuffle` không làm xáo `nums1`). Dữ liệu sinh với giá trị `0..4` để **thường xuyên có phần tử trùng** — chính là vùng dễ sai.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Ánh xạ ngược chiều (từ `nums2` sang `nums1`)

Đề hỏi: phần tử `i` của **`nums1`** nằm ở đâu trong **`nums2`**. Xây bảng từ `nums1` rồi tra `nums2` sẽ cho mảng "ngược".

In [18]:
def wrong_direction(nums1, nums2):
    pos = {v: i for i, v in enumerate(nums1)}     # SAI: bảng của nums1
    return [pos[x] for x in nums2]

a, c = [12, 28, 46, 32, 50], [50, 12, 32, 46, 28]
r = wrong_direction(a, c)
print("sai :", r, "hợp lệ?", is_valid_mapping(a, c, r), "| đúng:", mapping_bijective(a, c))

sai : [4, 0, 3, 2, 1] hợp lệ? False | đúng: [1, 4, 3, 2, 0]


Mẹo nhớ: **bảng tra được xây từ mảng mà ta cần lấy chỉ số** (`nums2`), rồi **duyệt mảng còn lại** (`nums1`) để hỏi.

### Lỗi 2: Kỳ vọng `dict` giữ mọi chỉ số khi trùng

`{v: j for j, v in enumerate(nums2)}` ghi đè → chỉ còn chỉ số cuối. Không sai với cách chấm "hợp lệ", nhưng sai nếu cần ghép một-một.

In [19]:
a, c = [1, 1], [1, 1]
r = mapping_dict(a, c)
print(r, "| hoán vị?", is_valid_mapping(a, c, r, True))

[1, 1] | hoán vị? False


### Lỗi 3: Đánh dấu "đã dùng" bằng cách sửa `nums2` (gán `None`) rồi gọi `index`

Vẫn ra hoán vị, nhưng `O(n²)` và **phá hủy input** của người gọi — tác dụng phụ khó lường.

In [20]:
def wrong_mutate(nums1, nums2):
    res = []
    for x in nums1:
        j = nums2.index(x)
        res.append(j)
        nums2[j] = None              # SỬA input!
    return res

data2 = [5, 5, 3, 5]
print(wrong_mutate([5, 3, 5, 5], data2), "| nums2 sau khi gọi:", data2)

[0, 2, 1, 3] | nums2 sau khi gọi: [None, None, None, None]


**Sửa:** nếu cần đánh dấu, làm trên **bản sao** (`nums2[:]`) — hoặc tốt hơn dùng Cách 3.

### Lỗi 4: Dùng `pop(0)` trong vòng lặp

`pos[x].pop(0)` đúng nhưng mỗi lần dời cả list → `O(k)`; khi mọi phần tử bằng nhau, tổng thành `O(n²)`. Dùng `pop()` (cuối list) hoặc `collections.deque.popleft()`.

### Lỗi 5: Test bằng `assert result == đáp_án_mẫu` khi có phần tử trùng

Một lời giải hoàn toàn đúng có thể trả `[1, 0]` thay vì `[0, 1]` → test "fail" oan. Với bài nhiều đáp án, luôn dùng **validator**.

In [21]:
r = mapping_bijective([1, 1], [1, 1])
print("kết quả:", r, "| == [0, 1]?", r == [0, 1], "| validator:", is_valid_mapping([1, 1], [1, 1], r, True))

kết quả: [1, 0] | == [0, 1]? False | validator: True


## 14. Mẹo nhận dạng pattern

- "Phần tử này nằm **ở đâu** trong mảng kia?" hỏi nhiều lần → **bảng băm giá trị → chỉ số** (tra ngược / inverted index).
- Có **trùng lặp** và cần ghép một-một → giá trị → **list/queue chỉ số**, lấy ra khi dùng.
- Hai mảng là **hoán vị/anagram** của nhau → có thể **ghép theo thứ hạng** sau khi sắp xếp (argsort).
- Đề nói "trả về đáp án bất kỳ" → viết **validator** kiểm tính chất thay vì so cứng.
- Bài cùng họ: 1 Two Sum (giá trị → chỉ số), 242 Valid Anagram, 49 Group Anagrams, 1331 Rank Transform of an Array, 2032.

## 15. Biến thể

### 15.1. Kiểm tra `nums2` có thật là anagram của `nums1` không (LeetCode 242 cho mảng)

So sánh bảng tần suất bằng `Counter`.

In [22]:
from collections import Counter

def is_anagram_array(a, b2):
    return Counter(a) == Counter(b2)

print(is_anagram_array([12, 28, 46], [46, 12, 28]), is_anagram_array([1, 1, 2], [1, 2, 2]))

True False


### 15.2. LeetCode 1331 — Rank Transform of an Array

Thay mỗi phần tử bằng **hạng** của nó (1 = nhỏ nhất, giá trị bằng nhau cùng hạng). Kết hợp sắp xếp + bảng băm giá trị → hạng.

In [23]:
def array_rank_transform(arr):
    rank = {v: k + 1 for k, v in enumerate(sorted(set(arr)))}
    return [rank[x] for x in arr]

print(array_rank_transform([40, 10, 20, 30]))       # [4, 1, 2, 3]
print(array_rank_transform([100, 100, 100]))        # [1, 1, 1]
print(array_rank_transform([37, 12, 28, 9, 100, 56, 80, 5, 12]))

[4, 1, 2, 3]
[1, 1, 1]
[5, 3, 4, 2, 8, 6, 7, 1, 3]


### 15.3. Hoán vị nghịch đảo

Nếu `mapping` là hoán vị, mảng `inv` với `inv[mapping[i]] = i` cho ánh xạ **ngược lại** (từ `nums2` về `nums1`). Đây chính là "Lỗi 1" được làm **đúng**.

In [24]:
a, c = [12, 28, 46, 32, 50], [50, 12, 32, 46, 28]
m = mapping_bijective(a, c)
inv = [0] * len(m)
for i, j in enumerate(m):
    inv[j] = i
print("mapping:", m, "| inv:", inv, "| inv hợp lệ (nums2 -> nums1)?", is_valid_mapping(c, a, inv, True))

mapping: [1, 4, 3, 2, 0] | inv: [4, 0, 3, 2, 1] | inv hợp lệ (nums2 -> nums1)? True


## 16. Bài tập

### Bài 1 — Tính tay
`nums1 = [7, 2, 7, 9]`, `nums2 = [9, 7, 2, 7]`. Viết `pos` theo Cách 3 và kết quả `mapping` (dùng `pop()` lấy phần tử cuối).

### Bài 2 — Bảng giá trị → chỉ số đầu tiên
Viết `first_positions(nums)` trả dict `giá trị → chỉ số xuất hiện ĐẦU TIÊN` (khác dict comprehension giữ chỉ số cuối).

In [25]:
def first_positions(nums: list[int]) -> dict[int, int]:
    # TODO
    pass

# assert first_positions([1, 7, 1, 7, 3]) == {1: 0, 7: 1, 3: 4}
# assert first_positions([]) == {}

### Bài 3 — Điền chỗ trống lời giải chính

In [26]:
def exercise_anagram_mappings(nums1: list[int], nums2: list[int]) -> list[int]:
    pos = defaultdict(list)
    # TODO 1: với mỗi (j, v) trong nums2, thêm j vào pos[v]
    result = []
    for x in nums1:
        # TODO 2: lấy (và xóa) một chỉ số của x, thêm vào result
        pass
    return result

# r = exercise_anagram_mappings([5, 3, 5, 5], [5, 5, 3, 5])
# assert is_valid_mapping([5, 3, 5, 5], [5, 5, 3, 5], r, True)

### Bài 4 — Ánh xạ giữ thứ tự cho phần tử trùng
Trả mapping là hoán vị sao cho **các bản sao của cùng một giá trị được ghép theo thứ tự xuất hiện**: bản sao thứ `k` của `x` trong `nums1` ↦ bản sao thứ `k` của `x` trong `nums2`. (Gợi ý: `pop()` lấy cuối — cần đổi.)

In [27]:
def stable_mapping(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO
    pass

# assert stable_mapping([5, 3, 5, 5], [5, 5, 3, 5]) == [0, 2, 1, 3]
# assert stable_mapping([1, 1], [1, 1]) == [0, 1]

### Bài 5 — Số bước tráo tối thiểu (mảng không trùng)
`nums2` là hoán vị của `nums1`, **không có phần tử trùng**. Tính số phép **hoán đổi hai phần tử** ít nhất để biến `nums1` thành `nums2`. Gợi ý: mapping là một hoán vị; đếm **chu trình**: đáp án = `n - số_chu_trình`.

In [28]:
def min_swaps(nums1: list[int], nums2: list[int]) -> int:
    # TODO
    pass

# assert min_swaps([1, 2, 3], [1, 2, 3]) == 0
# assert min_swaps([1, 2, 3], [2, 1, 3]) == 1
# assert min_swaps([12, 28, 46, 32, 50], [50, 12, 32, 46, 28]) == 3

### Bài 6 — Mini challenge: Group Anagrams cho chuỗi (LeetCode 49)
Nhóm các chuỗi là anagram của nhau. Trả list các nhóm theo thứ tự xuất hiện của phần tử đầu nhóm; trong nhóm giữ thứ tự gốc.

In [29]:
def group_anagrams(strs: list[str]) -> list[list[str]]:
    # TODO
    pass

# assert group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
# assert group_anagrams([""]) == [[""]]

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Liệt kê chỉ số theo giá trị | `pos = {9: [0], 7: [1, 3], 2: [2]}` | `pop()` lần lượt cho 7, 2, 7, 9 |
| 2 | Chỉ ghi khi key **chưa có** | `if v not in d` | hoặc `d.setdefault(v, j)` |
| 3 | Như Cách 3 | `pos[v].append(j)` | `result.append(pos[x].pop())` |
| 4 | Lấy chỉ số **đầu** list | `deque` + `popleft()`, hoặc đảo list trước | `pos[v] = deque(...)`; `pos[x].popleft()` |
| 5 | Theo `i -> mapping[i]` sẽ quay về `i` | Đếm chu trình bằng mảng `seen` | Mỗi chu trình dài `L` cần `L - 1` lần tráo |
| 6 | Hai chuỗi anagram ⟺ cùng chuỗi đã sắp xếp | `dict` khóa = `''.join(sorted(s))` | `groups.setdefault(key, []).append(s)` |

## 18. Đáp án tham khảo

**Bài 1:** `pos = {9: [0], 7: [1, 3], 2: [2]}`. `x=7` → pop 3; `x=2` → pop 2; `x=7` → pop 1; `x=9` → pop 0. `mapping = [3, 2, 1, 0]`.

In [30]:
print("Bài 1:", trace_mapping([7, 2, 7, 9], [9, 7, 2, 7]))
from collections import deque


def answer_first_positions(nums):
    d = {}
    for j, v in enumerate(nums):
        d.setdefault(v, j)
    return d


def answer_exercise(nums1, nums2):
    pos = defaultdict(list)
    for j, v in enumerate(nums2):
        pos[v].append(j)
    result = []
    for x in nums1:
        result.append(pos[x].pop())
    return result


def answer_stable_mapping(nums1, nums2):
    pos = defaultdict(deque)
    for j, v in enumerate(nums2):
        pos[v].append(j)
    return [pos[x].popleft() for x in nums1]


def answer_min_swaps(nums1, nums2):
    m = mapping_dict(nums1, nums2)          # không trùng -> đã là hoán vị
    n, seen, cycles = len(m), [False] * len(m), 0
    for i in range(n):
        if not seen[i]:
            cycles += 1
            k = i
            while not seen[k]:
                seen[k] = True
                k = m[k]
    return n - cycles


def answer_group_anagrams(strs):
    groups = {}
    for s in strs:
        groups.setdefault("".join(sorted(s)), []).append(s)
    return list(groups.values())


assert answer_first_positions([1, 7, 1, 7, 3]) == {1: 0, 7: 1, 3: 4}
assert answer_first_positions([]) == {}
r = answer_exercise([5, 3, 5, 5], [5, 5, 3, 5])
assert is_valid_mapping([5, 3, 5, 5], [5, 5, 3, 5], r, True)
assert answer_stable_mapping([5, 3, 5, 5], [5, 5, 3, 5]) == [0, 2, 1, 3]
assert answer_stable_mapping([1, 1], [1, 1]) == [0, 1]
assert answer_min_swaps([1, 2, 3], [1, 2, 3]) == 0
assert answer_min_swaps([1, 2, 3], [2, 1, 3]) == 1
assert answer_min_swaps([12, 28, 46, 32, 50], [50, 12, 32, 46, 28]) == 3
assert answer_group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
assert answer_group_anagrams([""]) == [[""]]
print("Tất cả đáp án PASS")

Đoạn A: pos = {9: [0], 7: [1, 3], 2: [2]}
i=0: x=7 | pos[x] trước=[1, 3] -> lấy 3 | pos[x] sau=[1] | mapping=[3]
i=1: x=2 | pos[x] trước=[2] -> lấy 2 | pos[x] sau=[] | mapping=[3, 2]
i=2: x=7 | pos[x] trước=[1] -> lấy 1 | pos[x] sau=[] | mapping=[3, 2, 1]
i=3: x=9 | pos[x] trước=[0] -> lấy 0 | pos[x] sau=[] | mapping=[3, 2, 1, 0]
Bài 1: [3, 2, 1, 0]
Tất cả đáp án PASS


In [31]:
import itertools

def brute_min_swaps(a, target):
    # BFS trên mọi trạng thái (chỉ cho mảng rất nhỏ) để kiểm chứng công thức n - số chu trình
    start, goal = tuple(a), tuple(target)
    dist = {start: 0}
    q = deque([start])
    while q:
        cur = q.popleft()
        if cur == goal:
            return dist[cur]
        for i, j in itertools.combinations(range(len(cur)), 2):
            nxt = list(cur); nxt[i], nxt[j] = nxt[j], nxt[i]; nxt = tuple(nxt)
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1
                q.append(nxt)

rng = random.Random(5)
for _ in range(200):
    a = rng.sample(range(10), rng.randint(1, 6))
    t = a[:]
    rng.shuffle(t)
    assert answer_min_swaps(a, t) == brute_min_swaps(a, t)
print("Bài 5: công thức n - số chu trình khớp BFS trên 200 hoán vị ngẫu nhiên")

Bài 5: công thức n - số chu trình khớp BFS trên 200 hoán vị ngẫu nhiên


Ghi chú bài 5: mapping `[1, 4, 3, 2, 0]` có các chu trình `0 → 1 → 4 → 0` (dài 3) và `2 → 3 → 2` (dài 2) → `5 - 2 = 3` lần tráo. `itertools.combinations(range(n), 2)` sinh mọi cặp `(i, j)` với `i < j`.

### Template nhớ nhanh + Checklist submit

```python
from collections import defaultdict

class Solution:
    def anagramMappings(self, nums1, nums2):
        pos = defaultdict(list)                  # bảng tra xây từ mảng cần lấy CHỈ SỐ (nums2)
        for j, v in enumerate(nums2):
            pos[v].append(j)                     # giữ MỌI chỉ số -> xử lý trùng
        return [pos[x].pop() for x in nums1]     # mỗi chỉ số dùng đúng một lần
```

**Câu thần chú:** *"tra ngược giá trị → chỉ số — xây từ nums2, hỏi bằng nums1 — trùng thì list + pop — test bằng validator."*

**Checklist trước khi submit:**
- [ ] Đúng chiều: `nums2[mapping[i]] == nums1[i]`?
- [ ] Bảng tra xây từ `nums2`, duyệt `nums1` để hỏi?
- [ ] Đã nghĩ tới phần tử trùng (list chỉ số + `pop()`, hoặc chấp nhận dùng lại chỉ số)?
- [ ] Không sửa `nums2` của người gọi?
- [ ] Dùng `pop()` (cuối) chứ không `pop(0)` trong vòng lặp?
- [ ] Test bằng **validator** cho các ca có trùng, không so cứng với một đáp án mẫu?